# 23 — Prompt Observability and Failure Diagnosis

## Scenario
A customer asks: "What is the current price of Product X?"
The model replies: "The price is $100."
The customer complains, saying the website lists it as $120.

**The Problem:** Why did the model fail? Is the prompt bad? Is the model hallucinating? 

**The Solution:** We must use **Telemetry Tracing** to isolate the fault. We cannot just log everything blindly because of PII (Personally Identifiable Information), so we must scrub the logs.

In [ ]:
import os
import time
import json
import re
from google import genai

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'


## Step 1: The PII Scrubber & Tracer

In the enterprise, you cannot log Social Security Numbers, emails, or phone numbers to your Observability dashboard. We build a simple regex scrubber and a simulated Tracer.

In [ ]:
def scrub_pii(text: str) -> str:
    # Scrub emails
    text = re.sub(r'\S+@\S+', '[REDACTED_EMAIL]', text)
    return text

class Tracer:
    def __init__(self):
        self.spans = []
        
    def start_span(self, name: str, input_data: str):
        return {"name": name, "input": scrub_pii(input_data), "start_time": time.time()}
        
    def end_span(self, span: dict, output_data: str):
        span["output"] = scrub_pii(output_data)
        span["duration_sec"] = time.time() - span["start_time"]
        self.spans.append(span)
        
    def print_trace(self):
        print(json.dumps(self.spans, indent=2))
        self.spans = []

tracer = Tracer()


## Step 2: The Application Logic

Our app does two things: retrieves context from a database, then generates a response.

In [ ]:
def run_qa_pipeline(user_query: str, db_context: str):
    # SPAN 1: Retrieve Context
    span_retrieve = tracer.start_span("Retrieve_Context", input_data=user_query)
    time.sleep(0.1) # Simulate DB lookup
    retrieved_text = db_context # We inject this for the simulation
    tracer.end_span(span_retrieve, output_data=retrieved_text)
    
    # SPAN 2: LLM Generation
    prompt = f"You are a helpful assistant. Answer based on context.\nContext: {retrieved_text}\nQuery: {user_query}"
    span_llm = tracer.start_span("LLM_Generation", input_data=prompt)
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )
    
    tracer.end_span(span_llm, output_data=response.text)
    return response.text


## Step 3: Scenario A (Success)

The customer asks a question, and the system works perfectly.

In [ ]:
good_db_context = "Product X costs $120. Contact admin@store.com for bulk orders."
query = "What is the price of Product X? Send invoice to user@gmail.com."

print("--- Output ---")
print(run_qa_pipeline(query, good_db_context))
print("\n--- Trace ---")
tracer.print_trace() # Notice the emails are redacted!


## Step 4: Scenario B (The Failure Diagnosis)

The customer asks the same question, but the model says $100. We inspect the trace to find out why.

In [ ]:
# The database team accidentally rolled back their table to last week's data
stale_db_context = "Product X costs $100. Contact admin@store.com for bulk orders."
query = "What is the price of Product X? Send invoice to user@gmail.com."

print("--- Output ---")
print(run_qa_pipeline(query, stale_db_context))
print("\n--- Trace ---")
tracer.print_trace()


## Conclusion

By looking at the trace in Scenario B, an engineer can look at the `Retrieve_Context` span and immediately see that the database returned `$100`.

The prompt wasn't bad. The model wasn't hallucinating. The upstream context was stale.

Without observability traces breaking the pipeline into discrete spans, you would have spent hours tweaking the prompt trying to fix a database error.